# Four FN-derived results that survive the $d \approx 10$ nm thickness issue

The Beebe absolute identification $V_T \approx \phi$ is inconsistent with the
FN slope at the physical CrSBr thickness of $\approx 10$ nm: a free-electron
$m^*$ gives $\phi \sim 50$ meV from the slope, against $\sim 280$ meV from
Beebe. The absolute $\phi$ and the lever arm $L$ are therefore **dropped**
from this notebook.

What survives the $d$-uncertainty is **ratio-based and qualitative**, and the
four results below support the two-faced crossover story of
[`interpretation_and_placement_CrSBr_Device2_v3.md`](../../../input/interpretation_and_placement_CrSBr_Device2_v3.md)
without ever invoking an absolute barrier height.

1. **Regime identification** at 20 K  — linear branch and $V_T$ minimum in $\ln(|I|/V^2)$ vs $1/V$ identify high-bias transport as field emission.
2. **Slope ratio**  $\phi_{\mathrm{AFM}}/\phi_{\mathrm{FM}} = (B_{\mathrm{AFM}}/B_{\mathrm{FM}})^{2/3}$  — independent of $d$ and $m^*$.
3. **$B_{\mathrm{FN}}(T)$ sign change** between $\sim 70$ and $\sim 100$ K — one of three independent signatures of the $\sim 80$ K crossover.
4. **Magnetic-state separation lives in the FN exponent**, $\ln(I_{\mathrm{FM}}/I_{\mathrm{AFM}}) \propto 1/V$ — the bias amplification of the MR is the slope **difference**, not an absolute barrier.

The $V_{\mathrm{peak}}(H)$ peak-finding analysis (linear-in-$m$, Jullière
rejection, $\Delta V_{\mathrm{peak}}^{c,b} \approx 185$, $189$ meV) does *not*
need FN and lives in [`Barrier_vs_canting_20K.ipynb`](Barrier_vs_canting_20K.ipynb).
A bridge slide at the end ties FN to it.


In [ ]:
from scripts.utils import setup_notebook, OKABE_ITO_CYCLE
PROJECT_ROOT, np, pd, plt, Path = setup_notebook()
from scripts.IV_Hscan_gaussian import load_dataframe

DATA_DIR  = PROJECT_ROOT / "output" / "IV_H_scans" / "dataframes" / "c_scans"
H_AFM_MAX = 0.10     # T
I_NOISE   = 1.0e-9   # A; SMU noise floor

# Per-T FM-saturation cutoffs. H_sat^c falls from ~2.20 T (20 K) toward
# ~1.45 T near T_N (Lopez-Paz 2022; manuscript Section 4). Each entry is
# set ~0.10 T below the local H_sat so the FM window is genuinely saturated.
H_FM_MIN_BY_T = {
    20: 2.10, 30: 2.05, 40: 1.95, 50: 1.85,
    60: 1.75, 70: 1.65, 80: 1.55, 90: 1.45, 100: 1.35,
}


def fm_cutoff(T):
    if T in H_FM_MIN_BY_T:
        return H_FM_MIN_BY_T[T]
    # Linear interpolation as fallback for any T not in the table.
    return max(2.20 + (1.45 - 2.20) * (T - 20) / 80.0 - 0.10, 1.20)


def load_T(T):
    return load_dataframe(DATA_DIR / f"IV_gaussian_{T}K.pkl")


def average_iv(df, mask, n_grid=401, V_min=0.005, V_max=0.95,
               current_col="current_smooth_asym"):
    """Average antisymmetric I(V) over rows matching `mask` on a uniform V grid.
    Falls back to `current_smooth` if `current_smooth_asym` is not present."""
    if current_col not in df.columns:
        current_col = "current_smooth"
    V_grid = np.linspace(V_min, V_max, n_grid)
    curves = []
    for _, r in df.loc[mask].iterrows():
        V = np.asarray(r["voltage_smooth"])
        I = np.asarray(r[current_col])
        order = np.argsort(V)
        V, I = V[order], I[order]
        m_pos = V > 0
        if m_pos.sum() < 10:
            continue
        curves.append(np.interp(V_grid, V[m_pos], I[m_pos],
                                left=np.nan, right=np.nan))
    if not curves:
        return V_grid, None, 0
    A = np.vstack(curves)
    return V_grid, np.nanmean(A, axis=0), A.shape[0]


def fn_transform(V, I, I_floor=I_NOISE):
    mask = (V > 0) & np.isfinite(I) & (np.abs(I) >= I_floor)
    Vp = V[mask]; Ip = I[mask]
    x = 1.0 / Vp; y = np.log(np.abs(Ip) / Vp**2)
    order = np.argsort(x)
    return x[order], y[order], Vp[order]


def find_local_min(y, smooth=5):
    if y.size < smooth + 2:
        return None
    pad = smooth // 2
    yk = np.array([np.mean(y[max(0, i-pad):min(len(y), i+pad+1)])
                   for i in range(len(y))])
    dy = np.diff(yk)
    for i in range(1, len(dy)):
        if dy[i-1] <= 0 < dy[i]:
            return i
    return None


def fit_fn(V, I, R2_thr=0.998, min_pts=10):
    """End-anchored linear-region fit on the FN plot.

    Grows a contiguous window from the high-V end (smallest 1/V) outward,
    keeping growing while R^2 stays above `R2_thr`. The last accepted window
    defines the FN slope. Returns None if no window meets the threshold (the
    cleanest no-FN-regime signal).
    """
    x, y, Vp = fn_transform(V, I)
    if len(x) < min_pts:
        return None
    best = None
    for i1 in range(min_pts, len(x) + 1):
        xs, ys = x[:i1], y[:i1]
        slope, intercept = np.polyfit(xs, ys, 1)
        yhat = slope * xs + intercept
        ss_res = float(np.sum((ys - yhat)**2))
        ss_tot = float(np.sum((ys - ys.mean())**2))
        R2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0
        if R2 >= R2_thr:
            best = (i1, slope, intercept, R2)
        else:
            break
    if best is None:
        return None
    i1, slope, intercept, R2 = best
    i_loc = find_local_min(y)
    V_T = float(Vp[i_loc]) if i_loc is not None else float("nan")
    return dict(B_FN=-float(slope), slope=float(slope), intercept=float(intercept),
                R2=float(R2), V_T=V_T, i_window=(0, i1),
                x=x, y=y, Vp=Vp)


def build_AFM_FM(T):
    df = load_T(T)
    V, I_AFM, n_A = average_iv(df, df["H"].abs() < H_AFM_MAX)
    _, I_FM,  n_F = average_iv(df, df["H"].abs() > fm_cutoff(T))
    return V, I_AFM, I_FM, n_A, n_F


print(f"Setup ok. c-axis pickles 3-160 K in {DATA_DIR.relative_to(PROJECT_ROOT)}")


## Overview — Fowler-Nordheim plots at all temperatures

Before the four-result analysis, a first look at the raw FN traces across
the full temperature range. Each curve is $\ln(|I|/V^2)$ vs $1/V$ for the
AFM endpoint ($|H_\mathrm{z}| < 0.10$ T, solid) and the FM endpoint (saturated,
dashed), averaged in each window with the 1 nA noise floor applied. Color
encodes temperature.

- **20-80 K** — the FN linear branch is present at every $T$. The trace
  falls roughly linearly with $1/V$ at small $1/V$, and the FM curves sit
  systematically *above* the AFM curves (more current at the same bias)
  because $B_{\mathrm{FN}}^{\mathrm{FM}} < B_{\mathrm{FN}}^{\mathrm{AFM}}$.
- **80-150 K** — the FN regime is progressively lost. As $T$ increases the
  slope flattens, the magnetic-state separation collapses, and above
  $\sim 100$ K both curves become monotonically increasing in $1/V$, i.e.
  high-bias transport is no longer field emission. This is the visual
  fingerprint of the $\sim 80$ K crossover that Slide 3 quantifies via the
  sign change of $B_{\mathrm{FN}}(T)$.


In [ ]:
T_LIST_LOW  = [20, 30, 40, 50, 60, 70, 80]
T_LIST_HIGH = [80, 90, 100, 110, 120, 130, 140, 150]


def plot_FN_overview(T_list, ylim=(-17, -12), xlim=(0, 20)):
    fig, (ax_afm, ax_fm) = plt.subplots(1, 2, figsize=(12, 5), dpi=300, sharey=True)
    for i, T in enumerate(T_list):
        try:
            V, I_AFM_T, I_FM_T, nA_T, nF_T = build_AFM_FM(T)
        except FileNotFoundError:
            continue
        color = OKABE_ITO_CYCLE[i % len(OKABE_ITO_CYCLE)]
        for ax, I in [(ax_afm, I_AFM_T), (ax_fm, I_FM_T)]:
            if I is None:
                continue
            m = (V > 0) & np.isfinite(I) & (np.abs(I) >= I_NOISE)
            if m.sum() < 5:
                continue
            x_ = 1.0 / V[m]
            y_ = np.log(np.abs(I[m]) / V[m]**2)
            order = np.argsort(x_)
            ax.plot(x_[order], y_[order], "-", color=color, lw=1.5,
                    label=f"{T} K")
    for ax, panel in [(ax_afm, r"AFM ($|H_\mathrm{z}|<0.10$ T)"),
                      (ax_fm,  r"FM (saturated)")]:
        ax.set_xlabel(r"$1/V$ (V$^{-1}$)")
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)
        ax.text(0.97, 0.97, panel, transform=ax.transAxes,
                ha="right", va="top",
                bbox=dict(facecolor="white", edgecolor="0.7", alpha=0.85,
                          boxstyle="round,pad=0.3"))
        ax.legend(loc="lower right", ncol=1)
    ax_afm.set_ylabel(r"$\ln(|I|/V^2)$")
    fig.tight_layout()
    plt.show()


plot_FN_overview(T_LIST_LOW,  ylim=(-17, -12))
plot_FN_overview(T_LIST_HIGH, ylim=(-17, -10))


## Field evolution of the FN plot at 20 K (c-axis)

This is the field-axis counterpart to the temperature-evolution overview
above: $T$ is fixed at 20 K and the data are binned along $|H_\mathrm{z}|$
instead of across temperature. The c-axis at 20 K is the hard-axis canting
axis with $H_\mathrm{sat}^\mathrm{c} \approx 2.20$ T, so $H_\mathrm{z} = 0$
sits in the AFM ground state, $|H_\mathrm{z}| \gtrsim 2.10$ T is saturated
FM, and intermediate fields trace the continuous canting trajectory between
them. Slides 1-4 below use only the AFM and FM endpoints; this overview
fills in the canted states in between. What to look for: the FN trace
should rotate and translate smoothly between the AFM and FM end curves,
with both the FN slope (steepness on the low-$1/V$ side) and the sub-FN
baseline shifting monotonically as the canting angle grows.

In [ ]:
df20 = load_T(20)
H_BIN_CENTERS = [0.00, 0.30, 0.60, 0.90, 1.20, 1.50, 1.80, 2.20]
bin_hw = 0.10

norm = plt.Normalize(vmin=min(H_BIN_CENTERS), vmax=max(H_BIN_CENTERS))
cmap = plt.cm.coolwarm   # blue (low H) -> red (high H)

fig, ax = plt.subplots(figsize=(6, 5), dpi=300)
rows = []
for Hc in H_BIN_CENTERS:
    if Hc == 0.0:
        mask = np.abs(df20["H"]) < bin_hw
    else:
        mask = np.abs(np.abs(df20["H"]) - Hc) < bin_hw
    V_grid, I_mean, n_curves = average_iv(df20, mask)
    if I_mean is None:
        rows.append((Hc, 0, np.nan, np.nan))
        continue
    x, y, Vp = fn_transform(V_grid, I_mean)
    color = cmap(norm(Hc))
    ax.plot(x, y, "-", color=color, lw=1.5)
    fr_ = fit_fn(V_grid, I_mean)
    if fr_ is not None and (Hc == 0.0 or Hc >= H_FM_MIN_BY_T[20]):
        i0, i1 = fr_["i_window"]
        xs = fr_["x"][i0:i1]
        ax.plot(xs, fr_["slope"] * xs + fr_["intercept"], ls=":",
                color=color, lw=2.0)
    B_FN = fr_["B_FN"] if fr_ is not None else np.nan
    V_T  = fr_["V_T"]  if fr_ is not None else np.nan
    rows.append((Hc, n_curves, B_FN, V_T))

sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label(r"$|H_\mathrm{z}|$ (T)")

ax.set_xlabel(r"$1/V$ (V$^{-1}$)")
ax.set_ylabel(r"$\ln(|I|/V^2)$")
#ax.set_xlim(0, 20)
#ax.set_ylim(-17, -12)
fig.tight_layout()
plt.show()

print(f"{'H_center (T)':>13}  {'N_curves':>9}  {'B_FN (V)':>10}  {'V_T (V)':>10}")
for Hc, n, B, VT in rows:
    B_str  = f"{B:10.3f}"  if np.isfinite(B)  else f"{'nan':>10}"
    VT_str = f"{VT:10.3f}" if np.isfinite(VT) else f"{'nan':>10}"
    print(f"{Hc:13.2f}  {n:9d}  {B_str}  {VT_str}")


## Slide 1 — Regime identification at 20 K

High-bias transport at 20 K is **Fowler-Nordheim field emission**, not Ohmic
nor direct tunneling. Two signatures:

1. A linear branch in $\ln(|I|/V^2)$ vs $1/V$ at small $1/V$ (high $V$).
2. A minimum $V_T$ separating the FN branch (high $V$, low $1/V$) from the
   sub-FN region (low $V$, high $1/V$).

The plot below shows both for AFM and FM endpoints at 20 K, with a noise
floor of 1 nA and a linear-fit threshold $R^2 > 0.998$. The slope and the
$V_T$ marker are read off here; **no absolute energy is claimed from this
slide**, only the existence of the FN regime.


In [ ]:
V20, I_AFM, I_FM, nA, nF = build_AFM_FM(20)
fn_AFM = fit_fn(V20, I_AFM)
fn_FM  = fit_fn(V20, I_FM)

fig, ax = plt.subplots(figsize=(6, 5), dpi=300)
for label, fr_, dcolor, fcolor, marker in [
    ("AFM", fn_AFM, OKABE_ITO_CYCLE[1], OKABE_ITO_CYCLE[6], "o"),
    ("FM",  fn_FM,  OKABE_ITO_CYCLE[5], OKABE_ITO_CYCLE[3], "s"),
]:
    ax.plot(fr_["x"], fr_["y"], marker, color=dcolor, ms=3, alpha=0.5,
            label=f"{label} data")
    i0, i1 = fr_["i_window"]
    xs = fr_["x"][i0:i1]
    ax.plot(xs, fr_["slope"] * xs + fr_["intercept"], ls=":",
            color=fcolor, lw=2.4, label=f"{label} FN linear fit")
    if np.isfinite(fr_["V_T"]):
        ax.axvline(1.0 / fr_["V_T"], color=fcolor, ls="--", lw=1.0, alpha=0.7)

ax.set_xlabel(r"$1/V$ (V$^{-1}$)")
ax.set_ylabel(r"$\ln(|I|/V^2)$")
ax.legend(loc="lower left")
fig.tight_layout()
plt.show()

print(f"20 K c-axis FN fits (|I| >= {I_NOISE*1e9:.1f} nA, R^2 > 0.998):")
for label, fr_ in [("AFM", fn_AFM), ("FM", fn_FM)]:
    Vlo = fr_["Vp"][fr_["i_window"][0]]
    Vhi = fr_["Vp"][fr_["i_window"][1] - 1]
    print(f"  {label}:  B_FN = {fr_['B_FN']:.3f} V,  V_T = {fr_['V_T']*1000:.0f} meV,  "
          f"R^2 = {fr_['R2']:.4f},  V window [{Vlo:.3f}, {Vhi:.3f}] V")


## Slide 2 — Slope ratio: the only quantitative FN number that survives

The FN slope is

$$B_{\mathrm{FN}} = \frac{4\, d\, \sqrt{2 m^*}\, \phi^{3/2}}{3 \hbar e}.$$

Both the barrier thickness $d$ and the effective mass $m^*$ are common to
the AFM and FM endpoints (the junction geometry does not change with
applied field), so the ratio

$$\frac{\phi_{\mathrm{AFM}}}{\phi_{\mathrm{FM}}} = \left(\frac{B_{\mathrm{AFM}}}{B_{\mathrm{FM}}}\right)^{2/3}$$

is **independent of $d$ and $m^*$**. This is the **only FN number that
survives the $d \approx 10$ nm issue**, and is the quantity to quote in the
manuscript in place of the Beebe-derived absolute $\phi$.


In [ ]:
ratio_B   = fn_AFM["B_FN"] / fn_FM["B_FN"]
ratio_phi = ratio_B**(2.0/3.0)
contrast  = (ratio_phi - 1.0) * 100.0

print(f"B_AFM           = {fn_AFM['B_FN']:.3f} V")
print(f"B_FM            = {fn_FM ['B_FN']:.3f} V")
print(f"B_AFM / B_FM    = {ratio_B:.3f}")
print(f"phi_AFM/phi_FM  = (B_AFM/B_FM)^(2/3) = {ratio_phi:.3f}")
print(f"fractional AFM-to-FM contrast       = {contrast:+.1f}%")


## Slide 5 — Bridge to the $V_{\mathrm{peak}}(H)$ analysis

Slides 1-4 establish that 20 K high-bias transport is field emission and
that the AFM-to-FM contrast lives in the FN exponent. The Beebe construction
further identifies the $(dI/dV)/(I/V)$ peak at $V \approx 2 V_T$ as a
**transition-voltage feature** — it sits at the FN crossover and tracks the
barrier-height contrast.

That identification — the bridge — is what justifies treating
$V_{\mathrm{peak}}(H)$ as a band-edge probe in
[`Barrier_vs_canting_20K.ipynb`](Barrier_vs_canting_20K.ipynb). The **shape**
of $V_{\mathrm{peak}}(H)$ (linear-in-$m$ , Jullière rejection), the
**endpoint contrast** $\Delta V_{\mathrm{peak}}^{\mathrm{c}}\approx 185$ meV and
$\Delta V_{\mathrm{peak}}^{\mathrm{b}}\approx 189$ meV, and the **temperature
evolution** $\Delta V_{\mathrm{peak}}(T)$ are all measured **without ever
invoking FN**. They are differential-conductance peak-fit results.

The role of this notebook is therefore:

- document the FN regime so the peak identification is physically grounded (slide 1),
- report the FN slope ratio as the $d$-independent fractional contrast (slide 2),
- demonstrate the $\sim 80$ K crossover via the slope sign change (slide 3),
- make explicit that the FN-exponent amplification drives the bias growth of MR (slide 4).

None of this requires the absolute $\phi$.


## Slide 6 — Summary: how the four FN results support the main story

| # | FN result | Robust to $d$, $m^*$? | Supports manuscript section |
|---|-----------|------------------------|------------------------------|
| 1 | Linear FN branch + $V_T$ minimum at 20 K | Qualitative | §3: identifies high-bias channel as field emission; justifies $V_{\mathrm{peak}}$ as a TVS feature |
| 2 | $\phi_{\mathrm{AFM}}/\phi_{\mathrm{FM}} = (B_{\mathrm{AFM}}/B_{\mathrm{FM}})^{2/3}$ | **Yes** ($d, m^*$ cancel) | §3 + §5: fractional AFM-to-FM contrast; consistent with a giant exchange-driven splitting on the ARPES scale (Watson 2024); replaces the Beebe absolute number as the load-bearing FN quantity |
| 3 | $B_{\mathrm{FN}}(T)$ sign change at 70-100 K | Qualitative | §6: one of three independent signatures of the $\sim 80$ K crossover (the others — $dI/dV$ peak loss and field-dependent $E_a$ — live in other notebooks) |
| 4 | $\ln(I_{\mathrm{FM}}/I_{\mathrm{AFM}}) \propto 1/V$ | Yes (slope difference) | §3 + §6: bias growth of the MR is the FN exponent acting on the slope difference, not an absolute-barrier effect |

**What is dropped on purpose:**

- The absolute $\phi_{\mathrm{AFM}}$, $\phi_{\mathrm{FM}}$ from $V_T \approx \phi$ (Beebe). At $d \approx 10$ nm with $m^* = m_e$ the FN slope is inconsistent with this identification by an order of magnitude. Either $d_{\mathrm{eff}} \ll d_{\mathrm{physical}}$ (an interfacial Schottky barrier rather than the full slab), or $m^*$ is anomalously small, or both. The FN data alone cannot distinguish these.
- The lever arm $L = \Delta V_{\mathrm{peak}}/\Delta \phi_{\mathrm{Beebe}}$. Inherits the Beebe assumption and shares its fate.

**Net effect on the paper.** The headline "$\sim 160$ meV calibrated splitting"
is replaced by "AFM-to-FM barrier-height **ratio** $\approx 1.58$ at 20 K, on
the few-hundred-meV interior scale established by ARPES (Watson 2024)." The
two-faced crossover story, the bias-robust $\sim 20$ meV edge splitting, and
the linear-in-$m$ vs $1 - m^2$ shape distinction are untouched and remain
the load-bearing claims.
